In [1]:
import bw2analyzer as ba
import bw2data as bd
import bw2calc as bc
import bw2io as bi
import matrix_utils as mu
import bw_processing as bp
import time
import wurst
import os
from pathlib import Path
import copy
import pandas as pd

In [2]:
start_time = time.time()

In [3]:
bd.projects.set_current('nitrous_oxide_production')

In [4]:
bd.databases

Databases dictionary with 10 object(s):
	ecoinvent-3.10-biosphere
	ecoinvent-3.10-cutoff
	ecoinvent-3.10-cutoff-Base-2020
	ecoinvent-3.10-cutoff-Base-2050
	ecoinvent-3.10-cutoff-RCP19-2050
	ecoinvent-3.10-cutoff-RCP26-2050
	nitrous-oxide-ei310-Base-2020
	nitrous-oxide-ei310-Base-2050
	nitrous-oxide-ei310-RCP19-2050
	nitrous-oxide-ei310-RCP26-2050

In [20]:
# import inventories from excel
excel_import = bi.ExcelImporter(os.path.join('..', 'data', 'inventories_ei310.xlsx'))
# excel_import = bi.ExcelImporter(os.path.join('..', 'data', 'inventories.xlsx'))
excel_import.apply_strategies()
excel_import.match_database('ecoinvent-3.10-cutoff-Base-2020', fields = ('name', 'reference product', 'unit', 'location')) # match flows from ecoinvent database
excel_import.match_database('ecoinvent-3.10-biosphere', fields = ('name', 'unit', 'categories')) # match flows from biosphere database
excel_import.match_database(fields = ('name', 'reference product', 'unit', 'location')) # match flows from new database
excel_import.statistics()

Extracted 5 worksheets in 0.04 seconds
Applying strategy: csv_restore_tuples
Applying strategy: csv_restore_booleans
Applying strategy: csv_numerize
Applying strategy: csv_drop_unknown
Applying strategy: csv_add_missing_exchanges_section
Applying strategy: normalize_units
Applying strategy: normalize_biosphere_categories
Applying strategy: normalize_biosphere_names
Applying strategy: strip_biosphere_exc_locations
Applying strategy: set_code_by_activity_hash
Applying strategy: link_iterable_by_fields
Applying strategy: assign_only_product_as_production
Applying strategy: link_technosphere_by_activity_hash
Applying strategy: drop_falsey_uncertainty_fields_but_keep_zeros
Applying strategy: convert_uncertainty_types_to_integers
Applying strategy: convert_activity_parameters_to_list
Applied 16 strategies in 10.76 seconds
Applying strategy: link_iterable_by_fields
Applying strategy: link_iterable_by_fields
Applying strategy: link_iterable_by_fields
21 datasets
	191 exchanges
	Links to the fo

(21, 191, 0, 0)

In [21]:
# modify inventories to substitute 'reference product' with 'product' in all the exchanges
for ds in excel_import:
    for exchange in ds['exchanges']:
        if 'reference product' in exchange:
            exchange['product'] = exchange.pop('reference product')

In [22]:
lci_base_db_name = 'nitrous-oxide-ei310-Base-2020'

In [23]:
new_locations = {'CN' : 'China',
                'US' : 'United States of America',
                'RER' : 'Western Euroape'}

In [24]:
global_db = excel_import.data # add all inventories
ecoinvent_base_db_name = 'ecoinvent-3.10-cutoff-Base-2020'
biosphere_db_name = 'ecoinvent-3.10-biosphere'
ecoinvent_db_dict = [ds.as_dict() for ds in bd.Database('ecoinvent-3.10-cutoff-Base-2020')] # convert ecoinvent database to dictionary
biosphere_db_dict = [ds.as_dict() for ds in bd.Database('ecoinvent-3.10-biosphere')] # convert biosphere database to dictionary # maybe not needed?

ds_to_regionalize = global_db

regionalized_ds = []
for ds in ds_to_regionalize:
    for loc in new_locations:
        dsCopy = wurst.transformations.geo.copy_to_new_location(ds, loc)
        regionalized_ds.append(dsCopy)

# Relink technosphere exchanges to the new locations
for ds in regionalized_ds:
    for exchange in ds['exchanges']:
        if exchange['type'] == 'technosphere':
            if exchange['database'] == ecoinvent_base_db_name:
                ds_output = [ds_instance for ds_instance in ecoinvent_db_dict if exchange['name'] == ds_instance['name'] 
                                and exchange['product'] == ds_instance['reference product'] 
                                and ds['location'] == ds_instance['location']]
                if not ds_output and 'market group for electricity' in exchange['name']:
                    exchange['name'] = exchange['name'].replace('group ', '')
                    ds_output = [ds_instance for ds_instance in ecoinvent_db_dict if exchange['name'] == ds_instance['name'] 
                                and exchange['product'] == ds_instance['reference product'] 
                                and ds['location'] == ds_instance['location']]
            elif exchange['database'] == lci_base_db_name:
                ds_output = [ds_instance for ds_instance in regionalized_ds if exchange['name'] == ds_instance['name']
                                and ds['location'] == ds_instance['location']]
            if ds_output:
                    exchange.update({'location': ds_output[0]['location']})

In [25]:
db = global_db + regionalized_ds
db_linked = copy.deepcopy(db)

production = lambda x : x['type'] == 'production'
technosphere = lambda x : x['type'] == 'technosphere'
biosphere = lambda x : x['type'] == 'biosphere'

# linking exchanges to database inventories
for ds in db_linked:
    for exchange in filter(production, ds['exchanges']):
        exchange.update({'input': (ds['database'], ds['code'])})
    for exchange in filter(technosphere, ds['exchanges']):
        try:
            exchange_link = wurst.get_one(db + ecoinvent_db_dict,
                                         wurst.equals('name', exchange['name']),
                                         wurst.equals('reference product', exchange['product']),
                                         wurst.equals('location', exchange['location']))
            exchange.update({'input' : (exchange_link['database'], exchange_link['code'])})
        except Exception:
            print(exchange['name'], exchange['product'], exchange['location'])
            raise
    # biosphere links maybe not needed
    for exchange in filter(biosphere, ds['exchanges']):
        if 'input' not in exchange:
            try:
                exchange_link = [ds['code'] for ds in biosphere_db_dict if ds['name'] == exchange['name'] and
                                                                        ds['unit'] == exchange['unit'] and
                                                                        ds['categories'] == exchange['categories']][0]
                exchange.update({'input': (biosphere_db_name, exchange_link)})
            except Exception:
                print(exchange['name'], exchange['unit'], exchange['categories'])
                raise

In [26]:
if lci_base_db_name in bd.databases:
    del bd.databases[lci_base_db_name]
wurst.write_brightway2_database(db_linked, lci_base_db_name) # write database

84 datasets
	764 exchanges
	Links to the following databases:
		ecoinvent-3.10-cutoff-Base-2020 (388 exchanges)
		nitrous-oxide-ei310-Base-2020 (204 exchanges)
		ecoinvent-3.10-biosphere (172 exchanges)
	0 unlinked exchanges (0 types)
		
10:22:24 [warning  ] Not able to determine geocollections for all datasets. This database is not ready for regionalization.


100%|██████████| 84/84 [00:00<00:00, 820.89it/s]


10:22:24 [info     ] Vacuuming database            
Created database: nitrous-oxide-ei310-Base-2020


In [27]:
base_db = wurst.extract_brightway2_databases('nitrous-oxide-ei310-Base-2020')

Getting activity data


100%|██████████| 84/84 [00:00<?, ?it/s]


Adding exchange data to activities


100%|██████████| 764/764 [00:00<00:00, 29896.42it/s]


Filling out exchange data


100%|██████████| 84/84 [00:00<00:00, 2099.48it/s]


In [28]:
database_names = bd.databases
ecoinvent_db_names = []
premise_db_names = []
for db_name in database_names:
    if ('Base' in db_name or 'RCP' in db_name) and 'nitrous-oxide' not in db_name and 'Base-2020' not in db_name:
        ecoinvent_db_names.append(db_name)
        premise_db_names.append(db_name.replace('ecoinvent-3.10-cutoff-', ''))
ecoinvent_db_names.sort()
premise_db_names.sort()

premise_db_names

['Base-2050', 'RCP19-2050', 'RCP26-2050']

In [29]:
biosphere_db = bd.Database('ecoinvent-3.10-biosphere') # import the biosphere database
biosphere_db_dict = [ds.as_dict() for ds in biosphere_db] # convert biosphere database to dictionary # maybe not needed?

In [30]:
for premise_db_name in premise_db_names:
    print('linking database ' + premise_db_name + '...')
    premise_db_dict = [ds.as_dict() for ds in bd.Database('ecoinvent-3.10-cutoff-' + premise_db_name)]

    db_linked = copy.deepcopy(base_db)

    production = lambda x : x['type'] == 'production'  
    technosphere = lambda x : x['type'] == 'technosphere'
    biosphere = lambda x : x['type'] == 'biosphere'

    for ds in db_linked:
        for exchange in filter(technosphere, ds['exchanges']):
            try:
                exchange_link = wurst.get_one(base_db + premise_db_dict,
                                            wurst.equals('name', exchange['name']),
                                            wurst.equals('reference product', exchange['product']),
                                            wurst.equals('location', exchange['location'])
                                            )
                exchange.update({'input' : (exchange_link['database'], exchange_link['code'])})
                exchange.update({'database' : exchange_link['database']})
            except Exception:
                print(exchange['name'], exchange['reference product'], exchange['location'])
                raise
        for exchange in filter(biosphere, ds['exchanges']):
            if 'input' not in exchange:
                try:
                    exchange_link = [ds['code'] for ef in biosphere_db_dict if ds['name'] == exchange['name'] and 
                                                                            ds['unit'] == exchange['unit'] and 
                                                                            ds['categories'] == exchange['categories']][0]
                    exchange.update({'input': ('biosphere3', exchange_link)})   
                except Exception:
                    print(exchange['name'], exchange['unit'], exchange['categories'])
                    raise
    db_name = 'nitrous-oxide-ei310-' + premise_db_name
    if db_name in bd.databases:
        del bd.databases[db_name]
    wurst.write_brightway2_database(db_linked, db_name)
    print(premise_db_name + ' linking and writing successful!')

linking database Base-2050...
84 datasets
	764 exchanges
	Links to the following databases:
		ecoinvent-3.10-cutoff-Base-2050 (388 exchanges)
		nitrous-oxide-ei310-Base-2050 (204 exchanges)
		ecoinvent-3.10-biosphere (172 exchanges)
	0 unlinked exchanges (0 types)
		
10:23:54 [warning  ] Not able to determine geocollections for all datasets. This database is not ready for regionalization.


100%|██████████| 84/84 [00:00<00:00, 760.82it/s]


10:23:54 [info     ] Vacuuming database            
Created database: nitrous-oxide-ei310-Base-2050
Base-2050 linking and writing successful!
linking database RCP19-2050...
84 datasets
	764 exchanges
	Links to the following databases:
		ecoinvent-3.10-cutoff-RCP19-2050 (388 exchanges)
		nitrous-oxide-ei310-RCP19-2050 (204 exchanges)
		ecoinvent-3.10-biosphere (172 exchanges)
	0 unlinked exchanges (0 types)
		
10:25:23 [warning  ] Not able to determine geocollections for all datasets. This database is not ready for regionalization.


100%|██████████| 84/84 [00:00<00:00, 745.73it/s]


10:25:23 [info     ] Vacuuming database            
Created database: nitrous-oxide-ei310-RCP19-2050
RCP19-2050 linking and writing successful!
linking database RCP26-2050...
84 datasets
	764 exchanges
	Links to the following databases:
		ecoinvent-3.10-cutoff-RCP26-2050 (388 exchanges)
		nitrous-oxide-ei310-RCP26-2050 (204 exchanges)
		ecoinvent-3.10-biosphere (172 exchanges)
	0 unlinked exchanges (0 types)
		
10:26:54 [warning  ] Not able to determine geocollections for all datasets. This database is not ready for regionalization.


100%|██████████| 84/84 [00:00<00:00, 631.10it/s]


10:26:54 [info     ] Vacuuming database            
Created database: nitrous-oxide-ei310-RCP26-2050
RCP26-2050 linking and writing successful!


In [31]:
acts = [ds for ds in bd.Database('nitrous-oxide-ei310-Base-2020') if 'hydrogen peroxide production, AO, green hydrogen' in ds['name']]
acts

['hydrogen peroxide production, AO, green hydrogen' (kilogram, GLO, None),
 'hydrogen peroxide production, AO, green hydrogen' (kilogram, CN, None),
 'hydrogen peroxide production, AO, green hydrogen' (kilogram, RER, None),
 'hydrogen peroxide production, AO, green hydrogen' (kilogram, US, None)]

In [32]:
for exc in acts[0].exchanges():
    if exc['type'] == 'technosphere':
        print(exc)

Exchange: 0.06585658549428021 kilogram 'hydrogen production, PEM electrolysis, green' (kilogram, GLO, None) to 'hydrogen peroxide production, AO, green hydrogen' (kilogram, GLO, None)>
Exchange: 0.0030874916157909655 kilogram 'market for anthraquinone' (kilogram, GLO, None) to 'hydrogen peroxide production, AO, green hydrogen' (kilogram, GLO, None)>


Exchange: 0.00193 kilogram 'market for benzene' (kilogram, RoW, None) to 'hydrogen peroxide production, AO, green hydrogen' (kilogram, GLO, None)>
Exchange: -0.000570169714215264 cubic meter 'market for wastewater, average' (cubic meter, RoW, None) to 'hydrogen peroxide production, AO, green hydrogen' (kilogram, GLO, None)>
Exchange: 0 kilogram 'market for palladium' (kilogram, GLO, None) to 'hydrogen peroxide production, AO, green hydrogen' (kilogram, GLO, None)>
Exchange: 0.08852518607292184 kilowatt hour 'market group for electricity, high voltage' (kilowatt hour, GLO, None) to 'hydrogen peroxide production, AO, green hydrogen' (kilogram, GLO, None)>
Exchange: 2.800425166921653 megajoule 'cooling water' (megajoule, GLO, None) to 'hydrogen peroxide production, AO, green hydrogen' (kilogram, GLO, None)>
Exchange: 1.9247124104337052 megajoule 'heat production, natural gas, at boiler condensing modulating >100kW' (megajoule, RoW, None) to 'hydrogen peroxide production, AO, green hydroge

In [33]:
method = ("IPCC 2021", "climate change", "GWP 100a, incl. H and bio CO2")

In [34]:
for act in acts:
    lca = bc.LCA({act: 1}, method = method)
    lca.lci()
    lca.lcia()
    print(act['location'], round(lca.score,3))

GLO 0.492
CN 0.51
RER 0.462
US 0.479
